# Querycart
I081 Dhruv Paleja · I053 Manan Shah

Open this notebook from the project folder. Python 3 and Node.js 22+ are required.

In [1]:
from pathlib import Path
import subprocess, json
root = Path.cwd()
if not (root / 'public/engine.js').exists():
    raise RuntimeError('Open the notebook in the project root.')
def run_node(code):
    r = subprocess.run(['node', '--input-type=module', '-e', code], cwd=root, capture_output=True, text=True, encoding='utf-8')
    if r.returncode: raise RuntimeError(r.stderr)
    return json.loads(r.stdout)

## Example
Follow the input through the structured output. The same JavaScript module is used by the deployed website.

In [2]:
example = run_node("import{readFileSync}from'node:fs';import{parseCSV,parseQuery,runQuery,sqlPreview}from'./public/engine.js';const data=parseCSV(readFileSync('public/inventory.csv','utf8'));const plan=parseQuery('Show Electronics under 1000',[...new Set(data.map(x=>x.category))]);console.log(JSON.stringify({plan,sql:sqlPreview(plan),rows:runQuery(plan,data)}));")
print(json.dumps(example, indent=2, ensure_ascii=False))

{
  "plan": {
    "intent": "list",
    "filters": [
      {
        "field": "category",
        "op": "eq",
        "value": "Electronics"
      },
      {
        "field": "price",
        "op": "lt",
        "value": 1000
      }
    ],
    "groupBy": null,
    "sort": null,
    "limit": null,
    "measure": null,
    "query": "Show Electronics under 1000"
  },
  "sql": {
    "sql": "SELECT sku, name, category, stock, reorder_level, price FROM inventory WHERE category = ? AND price < ?;",
    "params": [
      "Electronics",
      1000
    ]
  },
  "rows": [
    {
      "sku": "EL001",
      "name": "USB C charging cable",
      "category": "Electronics",
      "stock": 18,
      "reorder_level": 10,
      "price": 299
    },
    {
      "sku": "EL002",
      "name": "Wireless mouse",
      "category": "Electronics",
      "stock": 7,
      "reorder_level": 10,
      "price": 649
    },
    {
      "sku": "EL004",
      "name": "Laptop stand",
      "category": "Electronics",
     

## Evaluation
These original fixtures check the supported scope. They do not estimate performance on arbitrary real-world text. Failed examples remain in the result file.

In [3]:
r = subprocess.run(['node','scripts/evaluate.mjs'], cwd=root, capture_output=True, text=True, encoding='utf-8')
if r.returncode: raise RuntimeError(r.stderr)
metrics = json.loads((root/'public/evaluation.json').read_text(encoding='utf-8'))
print(json.dumps({k:v for k,v in metrics.items() if k not in ['results']}, indent=2))

{
  "type": "Original diagnostic fixture set, not arbitrary-language accuracy",
  "count": 32,
  "correct": 32,
  "supportedCount": 20,
  "supportedExact": 20,
  "rejectionCases": 12,
  "correctRejections": 12
}


## Observation
Grammar coverage is the main limitation. The interface rejects unsupported wording and shows the interpreted filters before the result is used.

Try changing the example. Explain one correct output and one failure in the viva.